In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (scatole, sfere, fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1
struct Sfera   { double cx, cy, cz, r; };                 // centro (cx, cy, cz) e raggio r


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
// Stanza = cubo [-1, 1]^3, z verticale (z = -1 pavimento). Fuori dal cubo = parete (INF).
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.8, -0.6, -0.6 };                    // partenza del drone: prima meta' della stanza, in basso
const Punto B = {  0.85, -0.4, -0.7 };                   // arrivo (sorgente del FIM): dietro l'armadio

// Mappa iniziale (mappa 0): muro con porta e finestra, armadio
const std::vector<Scatola> SCATOLE_0 = {                 // {x0, x1, y0, y1, z0, z1}
    { -0.05, 0.05, -1.0,  1.0, -1.0,  1.0 },             // muro a x = 0: divide la stanza in due
    {  0.35, 0.65, -0.7, -0.1, -1.0, -0.2 },             // armadio appoggiato a terra, oltre il muro
};
const Scatola PORTA    = { -0.05, 0.05, -0.3,  0.3, -1.0,  0.0 };       // porta nel muro: in basso, dal pavimento a meta' altezza
const Scatola FINESTRA = { -0.05, 0.05,  0.5,  0.8,  0.4,  0.8 };       // finestra nel muro: in alto
const Scatola SCATOLA_NUOVA = { 0.35, 0.65, 0.15, 0.55, 0.0, 0.45 };    // compare nella mappa 2, oltre il muro, sotto la finestra
const std::vector<Sfera> SFERE = {};                     // {cx, cy, cz, r}: nessuna sfera, per ora

// Ripianificazione: la mappa cambia durante il volo. imposta_mappa(m) sceglie la mappa attuale:
// 0 = iniziale, 1 = porta chiusa, 2 = porta chiusa + scatola nuova
inline std::vector<Scatola> SCATOLE = SCATOLE_0;
inline std::vector<Scatola> FORI = { PORTA, FINESTRA };  // zone in cui l'ostacolo viene tolto
inline void imposta_mappa(int m)
{
    SCATOLE = SCATOLE_0;
    FORI = (m == 0) ? std::vector<Scatola>{ PORTA, FINESTRA }    // porta aperta
                    : std::vector<Scatola>{ FINESTRA };          // da m = 1 la porta e' chiusa: resta solo la finestra
    if (m >= 2) SCATOLE.push_back(SCATOLA_NUOVA);                // da m = 2 compare la scatola nuova
}
// =====================================================================

// Vero se il punto (x, y, z) e' dentro la scatola s (bordi compresi)
inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

// Vero se il punto (x, y, z) e' dentro la sfera s (distanza dal centro <= raggio)
inline bool dentro(const Sfera& s, double x, double y, double z)
{
    return (x - s.cx) * (x - s.cx) + (y - s.cy) * (y - s.cy) + (z - s.cz) * (z - s.cz) <= s.r * s.r;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    for (const auto& s : SFERE)   if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // che toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sferadi raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                    margin_mask[(kk * N + ii) * N + jj] = 0;
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile fim3d.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>
#include <string>         // nomi dei file delle mappe         // salvataggio su file
#include "fmm3d.h"           // Fast Marching
#include "ambiente.h"        // Ambiente

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // (B) tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, const unsigned char* mask, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, mask, eps, Cb_step1);
    // (N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        const unsigned char* mask, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, mask, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, d_mask, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}
// FM2 completo (passata 1 + passata 2) sulla mappa attuale, con il FMM (CPU) o con il Block FIM (GPU).
// Restituisce u (tempo di arrivo da B), la F del FM2 e il tempo di calcolo delle due passate [ms].
void fm2(bool con_block_fim, int Nk, const std::vector<unsigned char>& mask, float eps, int niter,
         std::vector<float>& u, std::vector<float>& Fk, double& t_ms)
{
    const float hk = 2.0f / (Nk - 1);
    const int n = Nk * Nk * Nk;
    const std::vector<int> bordo = nodi_di_bordo(mask, Nk);                // sorgenti della passata 1
    const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };            // sorgente della passata 2
    std::vector<float> d(n, INF);
    for (int s : bordo) d[s] = 0.0f;
    const std::vector<float> uno(n, 1.0f);
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    if (con_block_fim) {                                     // GPU: tempo del solo ciclo dei giri (come nelle tabelle)
        float ms1 = 0.0f, ms2 = 0.0f;
        block_fim(d, uno, Nk, hk, bordo, mask, eps, niter, &ms1);
        Fk = lentezza_fm2(d);
        block_fim(u, Fk, Nk, hk, srcs, mask, eps, niter, &ms2);
        t_ms = ms1 + ms2;
    } else {                                                 // CPU: le due chiamate al FMM
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm(d, uno, Nk, hk, bordo, mask);
        Fk = lentezza_fm2(d);
        fmm(u, Fk, Nk, hk, srcs, mask);
        auto t1 = std::chrono::high_resolution_clock::now();
        t_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
    }
}

// Scrive i file di una mappa per Python: u, maschera, F e ambiente (ostacoli e fori della mappa attuale)
void scrivi_mappa(int Nk, int m, const std::vector<float>& u, const std::vector<unsigned char>& mask, const std::vector<float>& Fk)
{
    const std::string t = std::to_string(Nk) + "_mappa" + std::to_string(m);
    std::ofstream out("fim_" + t + ".bin", std::ios::binary);
    out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
    std::ofstream om("mask_" + t + ".bin", std::ios::binary);
    om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
    std::ofstream of("F_" + t + ".bin", std::ios::binary);
    of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
    std::ofstream amb("ambiente_mappa" + std::to_string(m) + ".txt");
    amb.precision(15);
    amb << "MARGINE " << MARGINE << "\n";
    amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
    amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
    for (const auto& s : SCATOLE) amb << "SCATOLA " << s.x0 << " " << s.x1 << " " << s.y0 << " " << s.y1 << " " << s.z0 << " " << s.z1 << "\n";
    for (const auto& s : SFERE)   amb << "SFERA " << s.cx << " " << s.cy << " " << s.cz << " " << s.r << "\n";
    for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 12;                                    // iterazioni locali massime (uscita anticipata): prove 8 / 24
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("    N   giri   max|FIM - FMM|   u(A) FIM   u(A) FMM   t FMM [ms]   t FIM [ms]   speedup\n");

    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo[-1, 1]^3
        const int n = Nk * Nk * Nk;                          // numero di nodi
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0= ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // la sorgente e' l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        // passata 1: distanza dagli ostacoli (F = 1), con FIM e FMM
        const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
        std::vector<float> d0(n, INF);                       // condizione iniziale: 0 sulbordo, INF altrove
        for (int s : bordo) d0[s] = 0.0f;
        const std::vector<float> uno(n, 1.0f);               // F = 1: u = distanza
        std::vector<float> d_ost, d_fmm;
        double t1_fmm = 1e30, t1_fim = 1e30;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu'veloce
            d_fmm = d0;
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(d_fmm, uno, Nk, hk, bordo, mask);            // CPU: passata 1 con il FMM
            auto t1 = std::chrono::high_resolution_clock::now();
            t1_fmm = std::min(t1_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());

            d_ost = d0;
            float ms = 0.0f;
            block_fim(d_ost, uno, Nk, hk, bordo, mask, eps, niter, &ms);   // GPU: passata 1 con il FIM
            t1_fim = std::min(t1_fim, (double)ms);
        }
        float diff1 = 0.0f;                                  // la distanza del FIM devecoincidere con quella del FMM
        for (int q = 0; q < n; ++q) diff1 = fmaxf(diff1, fabsf(d_ost[q] - d_fmm[q]));

        // FM2: lentezza F dalla distanza dagli ostacoli
        const std::vector<float> Fk = lentezza_fm2(d_ost);
        // passata 2: tempo di arrivo da B con questa F, con FMM e FIM
        std::vector<float> u0(n, INF);                       // condizione iniziale: 0 in B, INF altrove
        for (int s : srcs) u0[s] = 0.0f;

        std::vector<float> uk_fmm, uk_fim;
        double t_fmm = 1e30, t_fim = 1e30;
        int it = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            uk_fmm = u0;                                     // ogni misura riparte da capo
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(uk_fmm, Fk, Nk, hk, srcs, mask);             // CPU: passata 2 con il FMM (Alg. 3)
    auto t1 = std::chrono::high_resolution_clock::now();
            t_fmm = std::min(t_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());

            uk_fim = u0;
            float ms = 0.0f;
            it = block_fim(uk_fim, Fk, Nk, hk, srcs, mask, eps, niter, &ms);   // GPU:passata 2 con il Block FIM (Alg. 7)
            t_fim = std::min(t_fim, (double)ms);
        }

        // Controlli: FIM = FMM, nessun NaN, ostacoli a INF
        float diff = diff1;
        int n_nan = 0, ost_non_inf = 0;
        for (int q = 0; q < n; ++q) {
            diff = fmaxf(diff, fabsf(uk_fim[q] - uk_fmm[q]));
            if (std::isnan(uk_fim[q])) ++n_nan;
            if (!mask[q] && uk_fim[q] < INF) ++ost_non_inf;
        }
        if (n_nan > 0 || ost_non_inf > 0)                    // si stampa solo se qualcosa non va
            printf("ATTENZIONE (N = %d): %d valori NaN, %d nodi ostacolo raggiuntidall'onda\n", Nk, n_nan, ost_non_inf);

        const double t_fmm_tot = t1_fmm + t_fmm, t_fim_tot = t1_fim + t_fim;   // FM2 completo
        if (uk_fim[iA] >= INF)                               // l'onda partita da B non e' mai arrivata ad A
            printf("%5d   %4d   %14.2e   A NON RAGGIUNGIBILE da B\n", Nk, it, diff);
        else
            printf("%5d   %4d   %14.2e   %8.4f   %8.4f   %10.1f   %10.2f   %7.1f\n",
                   Nk, it, diff, uk_fim[iA], uk_fmm[iA], t_fmm_tot, t_fim_tot, t_fmm_tot / t_fim_tot);

        if (Nk == 129) {                                      // per i grafici (Python): soluzione, maschera e ambiente
            std::ofstream out("fim_129_ostacoli.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(uk_fim.data()), uk_fim.size() * sizeof(float));
            std::ofstream om("mask_129.bin", std::ios::binary);
            om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
            std::ofstream amb("ambiente.txt");               // l'ambiente di ambiente.h, una riga per elemento
            amb.precision(15);                               // 15 cifre: i numeri arrivano a Python esattamente come in ambiente.h
            amb << "MARGINE " << MARGINE << "\n";
            amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
            amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
            for (const auto& s : SCATOLE) amb << "SCATOLA " << s.x0 << " " << s.x1 << " " << s.y0 << " " << s.y1 << " " << s.z0 << " " << s.z1 << "\n";
            for (const auto& s : SFERE)   amb << "SFERA " << s.cx << " " << s.cy << " " << s.cz << " " << s.r << "\n";
            for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
            std::ofstream of("F_129.bin", std::ios::binary);  // la lentezza F usata nellapassata 2 (serve a Python)
            of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
        }
    }


    // RIPIANIFICAZIONE (N = 193): la mappa cambia durante il volo (0 = iniziale, 1 = porta chiusa,
    // 2 = porta chiusa + scatola nuova). Per ogni mappa: FM2 completo con FMM e Block FIM, tempi di ricalcolo
    // (minimo di 3 misure) e file per l'animazione in Python
    {
        const int Nr = 193;                                  // griglia della ripianificazione (h = 0.0104, cioe' 10 cm in una stanza di 20 m)
        std::ofstream tr("tempi_ripianificazione_" + std::to_string(Nr) + ".txt");   // una riga per mappa: m, t FMM [ms], t Block FIM [ms]
        printf("\nRipianificazione (N = %d): tempo per ricalcolare il FM2 (passata 1 + passata 2) dopo ogni cambio di mappa\n", Nr);
        printf("  mappa                       u(A) FMM   u(A) BFIM   max|FMM-BFIM|   t FMM [ms]   t BFIM [ms]   speedup\n");
        const char* nomi[3] = { "0 iniziale", "1 porta chiusa", "2 porta chiusa + scatola" };
        for (int m = 0; m < 3; ++m) {
            imposta_mappa(m);
            const std::vector<unsigned char> mask = costruisci_maschera(Nr);
            const int iA = nodo(Nr, A.x, A.y, A.z);
            std::vector<float> u_fmm, u_bfim, F_fmm, F_bfim;
            double t_fmm = 1e30, t_bfim = 1e30;
            for (int r = 0; r < ripetizioni; ++r) {
                double t;
                fm2(false, Nr, mask, eps, niter, u_fmm, F_fmm, t);   t_fmm = std::min(t_fmm, t);
                fm2(true,  Nr, mask, eps, niter, u_bfim, F_bfim, t); t_bfim = std::min(t_bfim, t);
            }
            float diff = 0.0f;
            for (size_t q = 0; q < u_fmm.size(); ++q) diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
            printf("  %-26s   %8.4f   %9.4f   %13.2e   %10.1f   %11.2f   %7.1f\n",
                   nomi[m], u_fmm[iA], u_bfim[iA], diff, t_fmm, t_bfim, t_fmm / t_bfim);
            tr << m << " " << t_fmm << " " << t_bfim << "\n";
            scrivi_mappa(Nr, m, u_bfim, mask, F_bfim);       // il drone del Block FIM e quello del FMM usano la stessa u
        }
        imposta_mappa(0);
    }
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 fim3d.cu -o fim3d && ./fim3d

In [ ]:
# RIPIANIFICAZIONE: due droni partono insieme da A; durante il volo la mappa cambia due volte
# (cambio 1: la porta si chiude; cambio 2: compare una scatola oltre il muro). A ogni cambio ogni drone si ferma
# finche' il suo algoritmo non ha ricalcolato il FM2 (tempi veri misurati da fim3d.cu), poi riparte da dove si trova.
# Blu = ripianifica con il Block FIM (GPU); arancione = con il FMM (CPU).
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import RegularGridInterpolator

# 0) PARAMETRI
N = 193                                                             # griglia dei file scritti da fim3d.cu (N_RIP)
METRI_PER_UNITA = 10.0                                              # il cubo [-1, 1]^3 e' una stanza di 20 m di lato
V_MAX = 2.0                                                         # velocita' massima del drone [m/s], dove F = 1
SEC_PER_U = METRI_PER_UNITA / V_MAX                                 # 1 unita' di u = 5 s di volo
FRAZ_CAMBIO_1 = 0.2                                                 # la porta si chiude dopo il 20% del volo previsto
X_CAMBIO_2 = 0.2                                                    # la scatola compare quando il drone blu supera x = 0.2
DT_FRAME = 0.25                                                     # un fotogramma ogni 0.25 s di volo
BLU, ARANCIO, ROSSO = "#2a78d6", "#eb6834", "#c0392b"
h = 2.0 / (N - 1)
x = -1.0 + h * np.arange(N)

# 1) DATI: le tre mappe (u da B, maschera, lentezza F, ostacoli) e i tempi di ricalcolo
def leggi_ambiente(nome):
    amb = {"SCATOLA": [], "SFERA": [], "FORO": []}
    for riga in open(nome):
        k, *v = riga.split(); v = [float(t) for t in v]
        if k in ("A", "B"): amb[k] = np.array(v)
        elif k == "MARGINE": amb[k] = v[0]
        else: amb[k].append(tuple(v))
    return amb
mappe = []
for m in range(3):
    t = f"{N}_mappa{m}"
    u = np.fromfile(f"fim_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)    # u[k, i, j] = u(z, y, x)
    libero = np.fromfile(f"mask_{t}.bin", dtype=np.uint8).reshape(N, N, N) == 1
    F = np.fromfile(f"F_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)
    mappe.append(dict(u=u, libero=libero, F=F, amb=leggi_ambiente(f"ambiente_mappa{m}.txt"),
                      F_i=RegularGridInterpolator((x, x, x), F), peso=RegularGridInterpolator((x, x, x), libero.astype(float))))
A, B = mappe[0]["amb"]["A"], mappe[0]["amb"]["B"]
tempi = {}                                                          # tempo di ricalcolo [s] del FM2 per mappa e metodo
for riga in open(f"tempi_ripianificazione_{N}.txt"):
    m, t_fmm, t_bfim = riga.split()
    tempi[int(m)] = {"FMM": float(t_fmm) / 1000.0, "Block FIM": float(t_bfim) / 1000.0}

# 2) PERCORSO da un punto qualsiasi su una mappa: discesa lungo u (26 vicini) + accorciamento
mosse = [(a, b, c) for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1) if (a, b, c) != (0, 0, 0)]
nodo = lambda p: tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
coord = lambda n: -1.0 + h * np.array(n[::-1], dtype=float)
def tempo(mp, p, q):                                                # tempo (unita' di u) lungo il segmento p-q: somma di F * ds
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.sum(mp["F_i"]((p + (t[:-1] + t[1:]) / 2 * (q - p))[:, ::-1])) * np.linalg.norm(q - p) / (len(t) - 1)
def si_vede(mp, p, q):                                              # il segmento p-q non attraversa ostacoli
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.all(mp["peso"]((p + t * (q - p))[:, ::-1]) > 0.999)
def percorso(mp, p0):
    u, libero, F = mp["u"], mp["libero"], mp["F"]
    n, nB = nodo(p0), nodo(B)
    if not libero[n] or u[n] >= 1e29:
        raise SystemExit(f"il drone in {np.round(p0, 2)} e' dentro un ostacolo o non raggiunge B: cambia la mappa")
    nodi = [n]
    while n != nB:
        vic = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
        vic = [w for w in vic if all(0 <= t < N for t in w) and libero[w]]
        n = min(vic, key=lambda w: u[w] + 0.5 * (F[n] + F[w]) * h * np.linalg.norm(np.subtract(w, n)))
        nodi.append(n)
    P = [np.asarray(p0, float)] + [coord(w) for w in nodi[1:]]
    tn = np.concatenate([[0], np.cumsum([tempo(mp, P[i], P[i + 1]) for i in range(len(P) - 1)])])
    out, a = [P[0]], 0
    while a < len(P) - 1:
        b = len(P) - 1
        while b > a + 1 and not (si_vede(mp, P[a], P[b]) and tempo(mp, P[a], P[b]) <= tn[b] - tn[a]):
            b -= 1
        out.append(P[b]); a = b
    return np.array(out)
def orario(mp, perc, t0):                                           # punti fitti del percorso e istante [s] di arrivo in ognuno
    s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(perc, axis=0), axis=1))])
    sf = np.linspace(0, s[-1], max(int(s[-1] / (h / 2)), 2))
    Pf = np.stack([np.interp(sf, s, perc[:, c]) for c in range(3)], axis=1)
    dt = mp["F_i"](((Pf[:-1] + Pf[1:]) / 2)[:, ::-1]) * np.diff(sf) * SEC_PER_U    # F * ds, in secondi
    return Pf, t0 + np.concatenate([[0], np.cumsum(dt)])

# 3) VOLO: il drone segue la mappa attuale; a ogni cambio si ferma per il tempo di ricalcolo del suo metodo
def vola(metodo, eventi):
    T, P, piani, fermo = [0.0], [A.copy()], [], []                  # piani: (istante, punti, orari) di ogni percorso calcolato
    pos, t, m = A.copy(), 0.0, 0
    for t_ev, m_nuova in eventi + [(np.inf, None)]:
        Pf, tf = orario(mappe[m], percorso(mappe[m], pos), t)
        piani.append((t, Pf, tf))
        prima = tf < t_ev
        T += list(tf[prima][1:]); P += list(Pf[prima][1:])
        if prima.all():                                             # arrivato a B prima del prossimo cambio
            return dict(T=np.array(T), P=np.array(P), piani=piani, fermo=fermo)
        k = np.searchsorted(tf, t_ev)                               # posizione all'istante del cambio
        a = (t_ev - tf[k - 1]) / (tf[k] - tf[k - 1])
        pos = Pf[k - 1] + a * (Pf[k] - Pf[k - 1])
        attesa = tempi[m_nuova][metodo]
        T += [t_ev, t_ev + attesa]; P += [pos, pos]; fermo.append((t_ev, t_ev + attesa))
        t, m = t_ev + attesa, m_nuova
T0 = orario(mappe[0], percorso(mappe[0], A), 0.0)[1][-1]           # durata del volo se la mappa non cambiasse
t1 = FRAZ_CAMBIO_1 * T0
vb = vola("Block FIM", [(t1, 1)])
t2 = vb["T"][np.argmax((vb["P"][:, 0] > X_CAMBIO_2) & (vb["T"] > t1))]
eventi = [(t1, 1), (t2, 2)]
voli = {met: vola(met, eventi) for met in ("Block FIM", "FMM")}
colore = {"Block FIM": BLU, "FMM": ARANCIO}
print(f"volo senza cambi: {T0:.2f} s | cambio 1 (porta chiusa) a t = {t1:.2f} s | cambio 2 (nuova scatola) a t = {t2:.2f} s")
for met, v in voli.items():
    att = [b - a for a, b in v["fermo"]]
    print(f"{met:9s}: arrivo in B dopo {v['T'][-1]:6.2f} s; fermo a ricalcolare {sum(att):6.3f} s "
          f"({' + '.join(f'{a * 1000:.1f} ms' for a in att)})")
rit = voli["FMM"]["T"][-1] - voli["Block FIM"]["T"][-1]
print(f"il drone del FMM arriva {rit:.2f} s dopo quello del Block FIM")

# 4) GEOMETRIA da disegnare: ogni ostacolo come parallelepipedi (il muro senza porta e finestra)
def pieni(b, fori):                                                 # la scatola b meno i fori: lista di parallelepipedi
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    out = []
    for x0, x1 in zip(tagli[0][:-1], tagli[0][1:]):
        for y0, y1 in zip(tagli[1][:-1], tagli[1][1:]):
            for z0, z1 in zip(tagli[2][:-1], tagli[2][1:]):
                c = ((x0 + x1) / 2, (y0 + y1) / 2, (z0 + z1) / 2)
                if not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori):
                    out.append((x0, x1, y0, y1, z0, z1))
    for asse in (2, 1, 0, 2, 1, 0):                                 # unisce i pezzi vicini: pochi parallelepipedi grandi
        out.sort(key=lambda p: tuple(p[2 * a] for a in range(3) if a != asse) + (p[2 * asse],))
        uniti = []
        for p in out:
            q = uniti[-1] if uniti else None
            if q and all(q[2 * a:2 * a + 2] == p[2 * a:2 * a + 2] for a in range(3) if a != asse) and abs(q[2 * asse + 1] - p[2 * asse]) < 1e-12:
                uniti[-1] = tuple(p[2 * a + s] if a == asse and s == 1 else q[2 * a + s] for a in range(3) for s in (0, 1))
            else:
                uniti.append(p)
        out = uniti
    return out
def mesh(scatole, nome, col, opacita):                              # un solo Mesh3d con tutti i parallelepipedi
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for b in scatole:
        o = len(X)
        X += [b[0], b[1], b[1], b[0], b[0], b[1], b[1], b[0]]; Y += [b[2], b[2], b[3], b[3]] * 2; Z += [b[4]] * 4 + [b[5]] * 4
        for i, j, k in [(0, 1, 2), (0, 2, 3), (4, 5, 6), (4, 6, 7), (0, 1, 5), (0, 5, 4), (1, 2, 6), (1, 6, 5), (2, 3, 7), (2, 7, 6), (3, 0, 4), (3, 4, 7)]:
            I.append(o + i); J.append(o + j); K.append(o + k)
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
a0, a1, a2 = (mappe[m]["amb"] for m in range(3))
porta = [f for f in a0["FORO"] if f not in a1["FORO"]][0]           # il foro che sparisce nella mappa 1
nuova = [s for s in a2["SCATOLA"] if s not in a1["SCATOLA"]][0]      # la scatola che compare nella mappa 2
muro = [p for p in pieni(a0["SCATOLA"][0], a0["FORO"])]           # il muro con la porta e la finestra (prima scatola)
altri = [p for s in a0["SCATOLA"][1:] for p in pieni(s, a0["FORO"])]  # gli altri ostacoli (l'armadio)
spigoli = []                                                        # i 12 spigoli della stanza
for a in (-1, 1):
    for b in (-1, 1):
        spigoli += [[(-1, a, b), (1, a, b)], [(a, -1, b), (a, 1, b)], [(a, b, -1), (a, b, 1)]]
stanza = go.Scatter3d(x=sum([[p[0][0], p[1][0], None] for p in spigoli], []), y=sum([[p[0][1], p[1][1], None] for p in spigoli], []),
                      z=sum([[p[0][2], p[1][2], None] for p in spigoli], []), mode="lines", line=dict(color="gray", width=2),
                      name="stanza (20 m)", hoverinfo="skip")
ab = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["  A (partenza)", "  B (arrivo)"],
                  marker=dict(size=8, color=["green", "black"]), textfont=dict(size=14), name="A e B")
def centro(b):                                                      # dove scrivere il nome di un ostacolo: sopra il centro
    return [(b[0] + b[1]) / 2], [(b[2] + b[3]) / 2], [b[5] + 0.08]

# 5) ANIMAZIONE 3D: ogni fotogramma ha tutte le tracce che cambiano, ognuna con il suo tipo
def compare(b, t_ev, t, nome):                                      # ostacolo che compare all'istante t_ev (prima: vuoto)
    if t < t_ev: return [go.Mesh3d(x=[], y=[], z=[], i=[], j=[], k=[], name=nome), go.Scatter3d(x=[], y=[], z=[], mode="text", name=nome)]
    cx, cy, cz = centro(b)
    return [mesh([b], nome, ROSSO, 0.8), go.Scatter3d(x=cx, y=cy, z=cz, mode="text", text=[nome], textfont=dict(size=14, color=ROSSO), name=nome)]
def drone(met, t):                                                  # scia percorsa, percorso previsto e drone all'istante t
    v = voli[met]; T, P = v["T"], v["P"]
    if t >= T[-1]: pos = P[-1]
    else:
        k = np.searchsorted(T, t, side="right"); a = (t - T[k - 1]) / max(T[k] - T[k - 1], 1e-12)
        pos = P[k - 1] + a * (P[k] - P[k - 1])
    fermo = any(a <= t < b for a, b in v["fermo"])
    piano = [p for p in v["piani"] if p[0] <= t][-1]
    resto = np.vstack([pos[None], piano[1][piano[2] > t]]) if not fermo and t < T[-1] else np.empty((0, 3))
    scia = np.vstack([P[T <= t], pos[None]])
    c = colore[met]
    return [go.Scatter3d(x=scia[:, 0], y=scia[:, 1], z=scia[:, 2], mode="lines", line=dict(color=c, width=8), name=f"{met}: percorso fatto"),
            go.Scatter3d(x=resto[:, 0], y=resto[:, 1], z=resto[:, 2], mode="lines", line=dict(color=c, width=3, dash="dash"), name=f"{met}: percorso previsto"),
            go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]], mode="markers", name=f"{met}: drone",
                         marker=dict(size=11 if fermo else 8, color="white" if fermo else c, line=dict(color=c, width=5)))]
def stato(met, t):
    v = voli[met]
    if t >= v["T"][-1]: return "arrivato"
    return "FERMO, ricalcola" if any(a <= t < b for a, b in v["fermo"]) else "in volo"
SCENA = dict(xaxis=dict(range=[-1.05, 1.05], autorange=False, title="x"), yaxis=dict(range=[-1.05, 1.05], autorange=False, title="y"),
             zaxis=dict(range=[-1.05, 1.05], autorange=False, title="z"), aspectmode="cube")
def dati(t):
    return compare(porta, t1, t, "porta chiusa") + compare(nuova, t2, t, "nuovo ostacolo") + drone("Block FIM", t) + drone("FMM", t)
def titolo(t):
    return (f"t = {t:4.1f} s   |   <span style='color:{BLU}'>Block FIM: {stato('Block FIM', t)}</span>   |   "
            f"<span style='color:{ARANCIO}'>FMM: {stato('FMM', t)}</span>")
t_fine = max(v["T"][-1] for v in voli.values())
istanti = np.append(np.arange(0.0, t_fine, DT_FRAME), t_fine)
statiche = [stanza, mesh(muro, "muro (porta e finestra aperte)", "lightgray", 0.4), mesh(altri, "armadio", "#a0785a", 0.7), ab]
mobili = list(range(len(statiche), len(statiche) + len(dati(0.0))))
fig = go.Figure(data=statiche + dati(0.0),
                frames=[go.Frame(data=dati(t), traces=mobili, name=f"{t:.2f}", layout=dict(title=dict(text=titolo(t)), scene=SCENA)) for t in istanti])
fig.update_layout(title=dict(text=titolo(0.0), x=0.5), scene=SCENA, scene_camera=dict(eye=dict(x=1.5, y=-1.7, z=1.0)),
                  height=800, margin=dict(l=0, r=0, t=60, b=90), legend=dict(x=0.0, y=0.95), uirevision="fisso",
                  updatemenus=[dict(type="buttons", direction="left", x=0.08, y=0.0, xanchor="right", yanchor="top", pad=dict(r=10, t=60), buttons=[
                      dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                      dict(label="Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
                  sliders=[dict(x=0.1, len=0.88, y=0.0, yanchor="top", currentvalue=dict(prefix="tempo di volo t = ", suffix=" s"), pad=dict(t=40),
                                steps=[dict(label=f"{t:.0f}" if abs(t - round(t)) < 1e-9 else "", method="animate",
                                            args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for t in istanti])])
fig.write_html("ripianificazione.html", include_plotlyjs="cdn")    # anche come file da scaricare: si apre nel browser
fig.show()

# 6) GRAFICO NEL TEMPO: quanta strada ha fatto ogni drone; le fasce sono le attese per il ricalcolo
g = go.Figure()
for met, v in voli.items():
    s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(v["P"], axis=0), axis=1))]) * METRI_PER_UNITA
    g.add_trace(go.Scatter(x=v["T"], y=s, mode="lines", line=dict(color=colore[met], width=3),
                           name=f"{met}: arriva a {v['T'][-1]:.1f} s"))
    for a, b in v["fermo"]:
        g.add_vrect(x0=a, x1=max(b, a + 0.05), fillcolor=colore[met], opacity=0.25, line_width=0)
for t_ev, nome in [(t1, "porta chiusa"), (t2, "nuovo ostacolo")]:
    g.add_vline(x=t_ev, line=dict(color=ROSSO, dash="dot"), annotation_text=nome, annotation_position="top left")
g.update_layout(template="plotly_white", height=420, title=f"Strada percorsa nel tempo: il drone del FMM arriva {rit:.1f} s dopo "
                f"(fasce = fermo a ricalcolare il percorso)", xaxis_title="tempo di volo [s]", yaxis_title="strada percorsa [m]",
                legend=dict(x=0.99, y=0.02, xanchor="right", yanchor="bottom"))
g.write_html("ripianificazione_tempi.html", include_plotlyjs="cdn")
g.show()
